In [1]:
import sys
import os

# On remonte d'un cran pour atteindre la racine du projet
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)


from src.O1_preprocessing.split_data import create_split
from src.O1_preprocessing.loader import get_available_patients

In [2]:
from src.utils.io import get_data_path
inventory = get_available_patients(str(get_data_path()))

2026-03-13 03:41:41,790 - src.O1_preprocessing.loader - INFO - 📂 Mode : Données BRUTES (raw) - Exploration
2026-03-13 03:41:41,793 - src.O1_preprocessing.loader - INFO - 📁 Volumes trouvés : 35 patients
2026-03-13 03:41:41,795 - src.O1_preprocessing.loader - INFO - 📁 Masques trouvés dans E:\Projet_fin_semestre\FUMPE_Detection\Projet_Detection_Embolie_Pulmonaire\data\raw\GroundTruth: 35 fichiers
2026-03-13 03:41:41,796 - src.O1_preprocessing.loader - INFO - 📊 Inventaire : 35 patients avec Volume + Masque.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 01 : Imports et Configuration
# ═══════════════════════════════════════════════════════════════
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

from src.O1_preprocessing.loader import get_available_patients, load_patient_data
from src.O2_classical_detection.detector import ClassicalDetector

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Environnement prêt pour la détection classique")



✅ Environnement prêt pour la détection classique


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1 : Imports et Configuration des Chemins
# ═══════════════════════════════════════════════════════════════
import sys
import os
from pathlib import Path

# ✅ Ajouter src/ au PYTHONPATH
src_path = Path(__file__).parent.parent / "src" if '__file__' in dir() else Path.cwd().parent / "src"
sys.path.insert(0, str(src_path))

# ✅ Utiliser ton utilitaire io.py
from src.utils.io import get_data_path, get_project_root

print(f"📂 Racine du projet : {get_project_root()}")
print(f"📁 Dossier data : {get_data_path()}")
#print(f"📁 Dossier src : {get_src_path()}")

# ✅ Vérifier que les dossiers existent
assert (get_data_path() / "raw").exists(), "❌ data/raw/ introuvable"
#assert (get_src_path() / "O1_preprocessing").exists(), "❌ src/O1_preprocessing/ introuvable"

print("✅ Tous les chemins sont valides !")

📂 Racine du projet : E:\Projet_fin_semestre\FUMPE_Detection\Projet_Detection_Embolie_Pulmonaire
📁 Dossier data : E:\Projet_fin_semestre\FUMPE_Detection\Projet_Detection_Embolie_Pulmonaire\data
✅ Tous les chemins sont valides !


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2 : Initialisation du Détecteur
# ═══════════════════════════════════════════════════════════════
config = {
    'lung_hu_threshold': -400,
    'threshold_method': 'percentile',
    'threshold_percentile': 95,
    'min_hu': 50,
    'max_hu': 400,
    'min_candidate_volume': 20,
    'max_candidate_volume': 5000
}

detector = ClassicalDetector(config)
print(f"🔧 Détecteur configuré : {config}")


🔧 Détecteur configuré : {'lung_hu_threshold': -400, 'threshold_method': 'percentile', 'threshold_percentile': 95, 'min_hu': 50, 'max_hu': 400, 'min_candidate_volume': 20, 'max_candidate_volume': 5000}


In [ ]:
# ═════════════════════════════════
# CELLULE 3 : Test sur un Patient 
# ══════════════════════════════════
from pathlib import Path
import sys
import os


NOTEBOOK_DIR = Path.cwd()  # Dossier actuel du notebook
PROJECT_ROOT = NOTEBOOK_DIR.parent  # Remonte d'un niveau depuis notebooks/
DATA_DIR = PROJECT_ROOT / "data"

# Vérifier que le dossier existe
if not DATA_DIR.exists():
    print(f"❌ Dossier data introuvable : {DATA_DIR}")
    print(f"📂 Dossier actuel : {NOTEBOOK_DIR}")
    print(f"💡 Essaye avec : DATA_DIR = Path('data') si ton notebook est à la racine")
else:
    print(f"✅ Dossier data trouvé : {DATA_DIR}")

# Lister le contenu pour debug
print(f"\n📁 Contenu de data/ :")
for item in DATA_DIR.iterdir():
    print(f"   - {item.name}")

# Charger l'inventaire avec le chemin absolu
inventory = get_available_patients(str(DATA_DIR))
print(f"\n📊 {len(inventory['with_both'])} patients disponibles")

if inventory['with_both']:
    test_id = inventory['with_both'][0]
    print(f"🧪 Test avec patient : {test_id}")
    
    # Charger les données (mode processed si disponible)
    vol, mask, meta = load_patient_data(test_id, inventory)
    
    if vol is not None:
        print(f"📦 Volume chargé : {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]")
        
        # Lancer la détection
        result = detector.detect(vol)
        
        print(f"\n🎯 Résultats :")
        print(f"   • Voxels pulmonaires : {result['lung_mask'].sum()}")
        print(f"   • Candidats détectés : {result['n_candidates']}")
        print(f"   • Voxels candidats : {result['total_candidate_voxels']}")
        
        # Afficher les features des candidats
        if len(result['candidates_df']) > 0:
            print(f"\n📋 Top 5 candidats par volume :")
            display(result['candidates_df'].nlargest(5, 'volume_voxels')[
                ['region_id', 'volume_voxels', 'intensity_mean', 'sphericity']
            ])
    else:
        print(f"❌ Échec de chargement du patient {test_id}")
else:
    print(f"\n❌ Aucun patient trouvé !")
    print(f"💡 Vérifie que :")
    print(f"   1. Le dossier data/raw/CT_scans_* existe")
    print(f"   2. Le dossier data/raw/GroundTruth/ existe")
    print(f"   3. Les dossiers PAT* contiennent des fichiers .dcm")

2026-03-13 03:01:09,854 - src.O1_preprocessing.loader - INFO - 📂 Mode : Données TRAITÉES (processed)
2026-03-13 03:01:09,860 - src.O1_preprocessing.loader - INFO - 📁 Volumes trouvés : 35 patients
2026-03-13 03:01:09,867 - src.O1_preprocessing.loader - INFO - 📊 Inventaire : 35 patients avec Volume + Masque.
2026-03-13 03:01:10,062 - src.O1_preprocessing.loader - INFO - ✅ Volume .npy chargé : (201, 512, 512)


✅ Dossier data trouvé : e:\Projet_fin_semestre\FUMPE_Detection\Projet_Detection_Embolie_Pulmonaire\data

📁 Contenu de data/ :
   - metadata
   - processed
   - raw
   - splits

📊 35 patients disponibles
🧪 Test avec patient : PAT001


2026-03-13 03:01:10,116 - src.O1_preprocessing.loader - INFO - ✅ Masque .npy chargé : (201, 512, 512)
2026-03-13 03:01:10,142 - src.O2_classical_detection.detector - INFO - 🚀 Démarrage de la détection classique...
2026-03-13 03:01:10,145 - src.O2_classical_detection.lung_segmentation - INFO - 🫁 Segmentation des poumons (méthode: threshold)


📦 Volume chargé : (201, 512, 512), HU: [0, 1]


2026-03-13 03:01:26,472 - src.O2_classical_detection.lung_segmentation - INFO - ✅ Masque pulmonaire: 0 voxels (0.0% du volume), 0 région(s)
2026-03-13 03:01:26,477 - src.O2_classical_detection.thresholding - INFO - 🔍 Seuillage adaptatif (méthode: percentile)
2026-03-13 03:01:32,889 - src.O2_classical_detection.thresholding - INFO -    Lissage gaussien (sigma=1.0)
2026-03-13 03:01:32,892 - src.O2_classical_detection.thresholding - WARNING - ⚠️ Aucune valeur dans le masque pulmonaire
2026-03-13 03:01:32,905 - src.O2_classical_detection.morphology - INFO - 🧹 Nettoyage morphologique des candidats...
2026-03-13 03:01:32,944 - src.O2_classical_detection.morphology - INFO -    Aucun candidat à nettoyer
2026-03-13 03:01:32,945 - src.O2_classical_detection.connected_components - INFO - 🔬 Extraction des caractéristiques 3D...
2026-03-13 03:01:32,979 - src.O2_classical_detection.connected_components - INFO -    Aucun candidat à extraire
2026-03-13 03:01:33,018 - src.O2_classical_detection.detecto


🎯 Résultats :
   • Voxels pulmonaires : 0
   • Candidats détectés : 0
   • Voxels candidats : 0


In [9]:
# Cellule 3 : Test sur un patient réel
inventory = get_available_patients("data")
if inventory['with_both']:
    test_id = inventory['with_both'][0]
    vol, mask, meta = load_patient_data(test_id, inventory)
    
    if vol is not None:
        result = detector.detect(vol, spacing=(1.0, 0.7, 0.7))
        print(f"🎯 {result['n_candidates']} candidats détectés")
        display(result['candidates_df'].head())

2026-03-13 02:13:30,922 - src.O1_preprocessing.loader - ERROR - ❌ Aucun dossier 'raw' ou 'processed' avec patients trouvé !


In [12]:
"""
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
from src.O1_preprocessing.loader import get_available_patients, load_patient_data
from src.O2_classical_detection.lung_segmentation import segment_lungs, visualize_lung_segmentation

# Charger un patient réel
inventory = get_available_patients('data')
if inventory['with_both']:
    vol, mask_gt, meta = load_patient_data(inventory['with_both'][0], inventory)
    
    print(f'📦 Volume: {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]')
    
    # Segmenter
    lung_mask = segment_lungs(vol, method='threshold', hu_threshold=-400)
    
    print(f'✅ Poumons: {lung_mask.sum()} voxels ({100*lung_mask.sum()/vol.size:.1f}%)')
    
    # Visualiser (optionnel, ouvre une fenêtre matplotlib)
    # visualize_lung_segmentation(vol, lung_mask, num_slices=5)

"""

"\nimport sys, os\nsys.path.insert(0, os.path.abspath('.'))\nimport numpy as np\nfrom src.O1_preprocessing.loader import get_available_patients, load_patient_data\nfrom src.O2_classical_detection.lung_segmentation import segment_lungs, visualize_lung_segmentation\n\n# Charger un patient réel\ninventory = get_available_patients('data')\nif inventory['with_both']:\n    vol, mask_gt, meta = load_patient_data(inventory['with_both'][0], inventory)\n    \n    print(f'📦 Volume: {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]')\n    \n    # Segmenter\n    lung_mask = segment_lungs(vol, method='threshold', hu_threshold=-400)\n    \n    print(f'✅ Poumons: {lung_mask.sum()} voxels ({100*lung_mask.sum()/vol.size:.1f}%)')\n    \n    # Visualiser (optionnel, ouvre une fenêtre matplotlib)\n    # visualize_lung_segmentation(vol, lung_mask, num_slices=5)\n\n"

In [22]:
# ═════════════════════════════════
# CELLULE 3 : Test sur un Patient (VERSION FINALE)
# ═════════════════════════════════
from pathlib import Path
import sys
import os

# ✅ Calculer le chemin vers le dossier 'data'
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

# Vérifier que le dossier existe
if not DATA_DIR.exists():
    print(f"❌ Dossier data introuvable : {DATA_DIR}")
else:
    print(f"✅ Dossier data trouvé : {DATA_DIR}")

# Lister le contenu pour debug
print(f"\n📁 Contenu de data/ :")
for item in DATA_DIR.iterdir():
    print(f"   - {item.name}")

# ✅ Charger l'inventaire (laisse le loader choisir raw/processed)
inventory = get_available_patients(str(DATA_DIR))

print(f"\n📊 {len(inventory['with_both'])} patients disponibles")
print(f"📂 Mode utilisé : {'PROCESSED' if inventory.get('use_processed', False) else 'RAW'}")

if inventory['with_both']:
    test_id = inventory['with_both'][0]
    print(f"🧪 Test avec patient : {test_id}")
    
    # Charger les données
    vol, mask_gt, meta = load_patient_data(test_id, inventory)
    
    if vol is not None:
        print(f"📦 Volume chargé : {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]")
        
        # ✅ Configuration avec seuils adaptés au mode
        if inventory.get('use_processed', False):
            # Mode PROCESSED : valeurs normalisées [0, 1]
            print("⚠️ Mode PROCESSED détecté - adaptation des seuils...")
            config = {
                'lung_hu_threshold': 0.17,   # ≈ -400 HU normalisé
                'threshold_method': 'percentile',
                'threshold_percentile': 95,
                'min_hu': 0.54,               # ≈ 50 HU normalisé
                'max_hu': 0.83,               # ≈ 400 HU normalisé
                'min_candidate_volume': 20,
                'max_candidate_volume': 5000
            }
            # Spacing par défaut (1mm isotrope si preprocessé correctement)
            spacing = (1.0, 1.0, 1.0)
        else:
            # Mode RAW : valeurs HU réelles
            config = {
                'lung_hu_threshold': -400,    # HU réels
                'threshold_method': 'percentile',
                'threshold_percentile': 95,
                'min_hu': 50,                  # HU réels
                'max_hu': 400,                 # HU réels
                'min_candidate_volume': 20,
                'max_candidate_volume': 5000
            }
            # Spacing depuis les métadonnées DICOM
            spacing = (
                meta.get('slice_spacing', 1.0),
                meta.get('spacing', [1.0, 1.0])[0],
                meta.get('spacing', [1.0, 1.0])[1]
            )
        
        detector = ClassicalDetector(config)
        
        # Lancer la détection
        result = detector.detect(vol, spacing=spacing)
        
        print(f"\n🎯 Résultats :")
        print(f"   • Voxels pulmonaires : {result['lung_mask'].sum()}")
        print(f"   • Candidats détectés : {result['n_candidates']}")
        print(f"   • Voxels candidats : {result['total_candidate_voxels']}")
        
        if len(result['candidates_df']) > 0:
            print(f"\n📋 Top 5 candidats par volume :")
            display(result['candidates_df'].nlargest(5, 'volume_voxels')[
                ['region_id', 'volume_voxels', 'intensity_mean', 'sphericity']
            ])
        else:
            print(f"\n💡 Aucun candidat détecté → Ajuster les seuils dans config")
    else:
        print(f"❌ Échec de chargement du patient {test_id}")
else:
    print(f"\n❌ Aucun patient trouvé !")

2026-03-13 03:04:53,995 - src.O1_preprocessing.loader - INFO - 📂 Mode : Données TRAITÉES (processed)
2026-03-13 03:04:53,997 - src.O1_preprocessing.loader - INFO - 📁 Volumes trouvés : 35 patients
2026-03-13 03:04:53,999 - src.O1_preprocessing.loader - INFO - 📊 Inventaire : 35 patients avec Volume + Masque.
2026-03-13 03:04:54,053 - src.O1_preprocessing.loader - INFO - ✅ Volume .npy chargé : (201, 512, 512)
2026-03-13 03:04:54,067 - src.O1_preprocessing.loader - INFO - ✅ Masque .npy chargé : (201, 512, 512)
2026-03-13 03:04:54,092 - src.O2_classical_detection.detector - INFO - 🚀 Démarrage de la détection classique...
2026-03-13 03:04:54,093 - src.O2_classical_detection.lung_segmentation - INFO - 🫁 Segmentation des poumons (méthode: threshold)


✅ Dossier data trouvé : e:\Projet_fin_semestre\FUMPE_Detection\Projet_Detection_Embolie_Pulmonaire\data

📁 Contenu de data/ :
   - metadata
   - processed
   - raw
   - splits

📊 35 patients disponibles
📂 Mode utilisé : PROCESSED
🧪 Test avec patient : PAT001
📦 Volume chargé : (201, 512, 512), HU: [0, 1]
⚠️ Mode PROCESSED détecté - adaptation des seuils...


2026-03-13 03:05:08,800 - src.O2_classical_detection.lung_segmentation - INFO - ✅ Masque pulmonaire: 50673762 voxels (96.2% du volume), 1 région(s)
2026-03-13 03:05:08,804 - src.O2_classical_detection.thresholding - INFO - 🔍 Seuillage adaptatif (méthode: percentile)
2026-03-13 03:05:24,654 - src.O2_classical_detection.thresholding - INFO -    Lissage gaussien (sigma=1.0)
2026-03-13 03:05:24,893 - src.O2_classical_detection.thresholding - INFO -    Seuil percentile 95: 0.9 HU
2026-03-13 03:05:24,953 - src.O2_classical_detection.thresholding - INFO -    Seuil appliqué: 0.9 - 0.83 HU
2026-03-13 03:05:26,781 - src.O2_classical_detection.thresholding - WARNING - ⚠️ Aucun candidat détecté
2026-03-13 03:05:26,796 - src.O2_classical_detection.morphology - INFO - 🧹 Nettoyage morphologique des candidats...
2026-03-13 03:05:26,831 - src.O2_classical_detection.morphology - INFO -    Aucun candidat à nettoyer
2026-03-13 03:05:26,832 - src.O2_classical_detection.connected_components - INFO - 🔬 Extra


🎯 Résultats :
   • Voxels pulmonaires : 50673762
   • Candidats détectés : 0
   • Voxels candidats : 0

💡 Aucun candidat détecté → Ajuster les seuils dans config


### Forcer le Mode RAW Sans grande Modification

In [6]:
# ═════════════════════════════════
# CELLULE 3 : Test sur un Patient 
# ═════════════════════════════════
from pathlib import Path
import sys
import os

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

# ✅ Cette ligne va maintenant charger en RAW car processed/ n'existe plus
inventory = get_available_patients(str(DATA_DIR))

print(f"\n📊 {len(inventory['with_both'])} patients disponibles")
print(f"📂 Mode utilisé : {'PROCESSED' if inventory.get('use_processed', False) else 'RAW'}")
# → Doit afficher : Mode utilisé : RAW ✅

2026-03-13 03:42:18,770 - src.O1_preprocessing.loader - INFO - 📂 Mode : Données BRUTES (raw) - Exploration
2026-03-13 03:42:18,772 - src.O1_preprocessing.loader - INFO - 📁 Volumes trouvés : 35 patients
2026-03-13 03:42:18,774 - src.O1_preprocessing.loader - INFO - 📁 Masques trouvés dans e:\Projet_fin_semestre\FUMPE_Detection\Projet_Detection_Embolie_Pulmonaire\data\raw\GroundTruth: 35 fichiers
2026-03-13 03:42:18,775 - src.O1_preprocessing.loader - INFO - 📊 Inventaire : 35 patients avec Volume + Masque.



📊 35 patients disponibles
📂 Mode utilisé : RAW


In [8]:
# ═════════════════════════════════
# CELLULE 3 : Test sur un Patient 
# ═════════════════════════════════
from pathlib import Path
import sys
import os

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

if not DATA_DIR.exists():
    print(f"❌ Dossier data introuvable : {DATA_DIR}")
else:
    print(f"✅ Dossier data trouvé : {DATA_DIR}")

print(f"\n📁 Contenu de data/ :")
for item in DATA_DIR.iterdir():
    print(f"   - {item.name}")

# ✅ Charger l'inventaire (mode RAW car processed_backup)
inventory = get_available_patients(str(DATA_DIR))

print(f"\n📊 {len(inventory['with_both'])} patients disponibles")
print(f"📂 Mode utilisé : {'PROCESSED' if inventory.get('use_processed', False) else 'RAW'}")

if inventory['with_both']:
    test_id = inventory['with_both'][0]
    print(f"🧪 Test avec patient : {test_id}")
    
    # ✅ Charger les données RAW
    vol, mask_gt, meta = load_patient_data(test_id, inventory)
    
    if vol is not None:
        print(f"📦 Volume chargé : {vol.shape}, HU: [{vol.min():.0f}, {vol.max():.0f}]")
        
        # ✅ Configuration avec seuils HU RÉELS
        config = {
            'lung_hu_threshold': -400,
            'threshold_method': 'percentile',
            'threshold_percentile': 85,
            'min_hu': 20,
            'max_hu': 400,
            'min_candidate_volume': 20,
            'max_candidate_volume': 5000
        }
        detector = ClassicalDetector(config)
        
        # ✅ Spacing depuis métadonnées
        spacing = (
            meta.get('slice_spacing', 1.0),
            meta.get('spacing', [1.0, 1.0])[0],
            meta.get('spacing', [1.0, 1.0])[1]
        )
        result = detector.detect(vol, spacing=spacing)
        
        print(f"\n🎯 Résultats :")
        print(f"   • Voxels pulmonaires : {result['lung_mask'].sum()}")
        print(f"   • Candidats détectés : {result['n_candidates']}")
        print(f"   • Voxels candidats : {result['total_candidate_voxels']}")
        
        if len(result['candidates_df']) > 0:
            print(f"\n📋 Top 5 candidats par volume :")
            display(result['candidates_df'].nlargest(5, 'volume_voxels')[
                ['region_id', 'volume_voxels', 'intensity_mean', 'sphericity']
            ])
        else:
            print(f"\n💡 Aucun candidat détecté → Ajuster les seuils")
    else:
        print(f"❌ Échec de chargement du patient {test_id}")
else:
    print(f"\n❌ Aucun patient trouvé !")

2026-03-13 03:48:51,615 - src.O1_preprocessing.loader - INFO - 📂 Mode : Données BRUTES (raw) - Exploration
2026-03-13 03:48:51,618 - src.O1_preprocessing.loader - INFO - 📁 Volumes trouvés : 35 patients
2026-03-13 03:48:51,619 - src.O1_preprocessing.loader - INFO - 📁 Masques trouvés dans e:\Projet_fin_semestre\FUMPE_Detection\Projet_Detection_Embolie_Pulmonaire\data\raw\GroundTruth: 35 fichiers
2026-03-13 03:48:51,620 - src.O1_preprocessing.loader - INFO - 📊 Inventaire : 35 patients avec Volume + Masque.


✅ Dossier data trouvé : e:\Projet_fin_semestre\FUMPE_Detection\Projet_Detection_Embolie_Pulmonaire\data

📁 Contenu de data/ :
   - metadata
   - processed_backup
   - raw
   - splits

📊 35 patients disponibles
📂 Mode utilisé : RAW
🧪 Test avec patient : PAT001


2026-03-13 03:48:52,260 - src.O1_preprocessing.loader - INFO - ✅ Volume chargé : (201, 512, 512) de PAT001
2026-03-13 03:48:52,424 - src.O1_preprocessing.loader - INFO - ✅ Masque chargé avec clé 'Mask'
2026-03-13 03:48:52,427 - src.O1_preprocessing.loader - INFO - Transpose appliqué: → (201, 512, 512)
2026-03-13 03:48:52,427 - src.O1_preprocessing.loader - INFO - ✅ Masque chargé : (201, 512, 512)
2026-03-13 03:48:52,468 - src.O2_classical_detection.detector - INFO - 🚀 Démarrage de la détection classique...
2026-03-13 03:48:52,469 - src.O2_classical_detection.lung_segmentation - INFO - 🫁 Segmentation des poumons (méthode: threshold)


📦 Volume chargé : (201, 512, 512), HU: [-1024, 3071]


2026-03-13 03:49:08,985 - src.O2_classical_detection.lung_segmentation - INFO - ✅ Masque pulmonaire: 11169045 voxels (21.2% du volume), 2 région(s)
2026-03-13 03:49:08,988 - src.O2_classical_detection.thresholding - INFO - 🔍 Seuillage adaptatif (méthode: percentile)
2026-03-13 03:49:18,754 - src.O2_classical_detection.thresholding - INFO -    Lissage gaussien (sigma=1.0)
2026-03-13 03:49:18,836 - src.O2_classical_detection.thresholding - INFO -    Seuil percentile 85: -884.6 HU
2026-03-13 03:49:18,913 - src.O2_classical_detection.thresholding - INFO -    Seuil appliqué: 20.0 - 400 HU
2026-03-13 03:49:21,568 - src.O2_classical_detection.thresholding - INFO - ✅ 115 voxels candidats détectés (0.000% du volume)
2026-03-13 03:49:21,569 - src.O2_classical_detection.thresholding - INFO -    5 régions candidates identifiées
2026-03-13 03:49:22,243 - src.O2_classical_detection.thresholding - INFO -    Tailles: min=12, moy=23, max=39
2026-03-13 03:49:22,260 - src.O2_classical_detection.morpholog


🎯 Résultats :
   • Voxels pulmonaires : 11169045
   • Candidats détectés : 3
   • Voxels candidats : 87

📋 Top 5 candidats par volume :


,region_id,volume_voxels,intensity_mean,sphericity
2,3,39.0,129.435898,0
1,2,28.0,183.750000,0
0,1,20.0,299.700012,0


#### Remaque:

Cela signifie que les "aspérités" ont ete bien supprimé aux bords des vaisseaux pour ne garder que le cœur des structures denses.